# Generated Project Anatomy

Inspect a fresh standalone wave-equation project generated by NRPy. This notebook identifies important generated files, rebuilds from a clean state, runs once, and locates the diagnostic output.

[Index](../index.ipynb) | Previous: [NRPy Package Map](package_map.ipynb) | Next: [NRPy in Ten Minutes](../1-intro/ten_minute_overview.ipynb)

## Why This Matters

Generated projects are the bridge between symbolic equations and compiled numerical programs. Inspecting one early makes later build and diagnostic steps easier to read.

## What You Will See

- A fresh generated Cartesian wave project.
- A project-relative file inventory and responsibility map.
- A build, run, parameter-file sample, and diagnostic preview.

## Table of Contents

1. Step 1: Verify the imported package and generator capability
2. Step 2: Create a fresh generation workspace
3. Step 3: Generate the Cartesian wave-equation project
4. Step 4: Inventory the generated tree
5. Step 5: Connect files to responsibilities
6. Step 6: Read the parameter file and command-line override path
7. Step 7: Inspect the `Makefile`, build, clean, and rebuild
8. Step 8: Run with a convergence-factor override and inspect diagnostics
9. Step 9: What to edit, what to regenerate, and where to go next

## Step 1: Verify the Imported Package and Generator Capability

This notebook needs an importable `nrpy` package and the Cartesian wave-equation generator module. The generator is a project entry point, so this check verifies that it is discoverable without importing it.

In [1]:
import importlib.util
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import nrpy

PROJECT_NAME = "wave_equation_cartesian"
GENERATOR_MODULE = "nrpy.examples.wave_equation_cartesian"

print("nrpy import succeeded")
generator_spec = importlib.util.find_spec(GENERATOR_MODULE)
print(f"{GENERATOR_MODULE}:", generator_spec is not None)

nrpy import succeeded
nrpy.examples.wave_equation_cartesian: True


## Step 2: Create a Fresh Generation Workspace

The generator writes `project/wave_equation_cartesian/` relative to its working directory and recreates that project directory. A fresh temporary workspace keeps existing learner projects untouched. The temporary-directory object is stored so the generated project remains available for the full notebook session.

In [2]:
_workspace = tempfile.TemporaryDirectory(prefix="nrpy-generated-project-")
WORKSPACE = Path(_workspace.name).resolve()
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME

print("workspace: temporary directory")
print(f"project directory: project/{PROJECT_NAME}")


def display_path(path):
    path = Path(path)
    if path == WORKSPACE:
        return "temporary workspace"
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        if path == PROJECT_DIR:
            return f"project/{PROJECT_NAME}"
        return path.name


def display_command(command):
    return " ".join("python" if str(part) == sys.executable else str(part) for part in command)


def _clean_text(text):
    text = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text)
    text = text.replace("\r", "\n")
    for raw, replacement in [
        (str(WORKSPACE), "temporary workspace"),
        (str(PROJECT_DIR), f"project/{PROJECT_NAME}"),
        (sys.executable, "python"),
    ]:
        text = text.replace(raw, replacement)
    return "\n".join(line.rstrip() for line in text.splitlines() if line.strip())


def run_command(command, cwd, max_lines=30):
    cwd = Path(cwd)
    print(f"$ {display_command(command)}")
    print(f"cwd: {display_path(cwd)}")
    completed = subprocess.run(
        command,
        cwd=cwd,
        check=False,
        capture_output=True,
        text=True,
    )
    print(f"return code: {completed.returncode}")
    output = _clean_text(completed.stdout + completed.stderr)
    if output:
        lines = output.splitlines()
        if len(lines) > max_lines:
            lines = lines[: max_lines // 2] + ["..."] + lines[-max_lines // 2 :]
        print("\n".join(lines))
    if completed.returncode != 0:
        raise RuntimeError(f"command failed: {' '.join(str(part) for part in command)}")
    return completed


def require_tool(name):
    path = shutil.which(name)
    if path is None:
        raise RuntimeError(f"required command not found: {name}")
    print(f"{name}: available")
    return path


def preview_file(path, max_lines=12):
    path = Path(path)
    print(path.name)
    for line in path.read_text(encoding="utf-8").splitlines()[:max_lines]:
        print(line.rstrip())

workspace: temporary directory
project directory: project/wave_equation_cartesian


## Step 3: Generate the Cartesian Wave-Equation Project

Run the generator explicitly as a module. A successful run creates `project/wave_equation_cartesian/` under the temporary workspace.

In [3]:
run_command([sys.executable, "-m", GENERATOR_MODULE], cwd=WORKSPACE)
print(f"Generated project: project/{PROJECT_NAME}")

$ python -m nrpy.examples.wave_equation_cartesian
cwd: temporary workspace


return code: 0
Finished! Now go into project/wave_equation_cartesian and type `make` to build, then ./wave_equation_cartesian to run.
    Parameter file can be found in wave_equation_cartesian.par
Generated project: project/wave_equation_cartesian


## Step 4: Inventory the Generated Tree

A generated BHaH project contains C sources, headers, a `Makefile`, a parameter file, and helper subdirectories. These files are outputs of the Python generator and can be regenerated.

In [4]:
required_relative_paths = [
    "Makefile",
    "wave_equation_cartesian.par",
    "BHaH_defines.h",
    "BHaH_function_prototypes.h",
    "main.c",
    "rhs_eval.c",
    "initial_data.c",
    "diagnostics.c",
    "numerical_grids_and_timestep.c",
    "cmdline_input_and_parfile_parser.c",
    "MoL/MoL_step_forward_in_time.c",
]

missing = [relpath for relpath in required_relative_paths if not (PROJECT_DIR / relpath).is_file()]
if missing:
    raise FileNotFoundError(f"missing generated file(s): {missing}")

c_sources = sorted(PROJECT_DIR.rglob("*.c"))
headers = sorted(PROJECT_DIR.rglob("*.h"))
print(f"C sources: {len(c_sources)}")
print(f"Headers: {len(headers)}")

print("Required files:")
for relpath in required_relative_paths:
    print(f"  {relpath}")

print("Generated file sample:")
for path in sorted(p.relative_to(PROJECT_DIR) for p in PROJECT_DIR.rglob("*") if p.is_file())[:25]:
    print(f"  {path}")

C sources: 15
Headers: 6
Required files:
  Makefile
  wave_equation_cartesian.par
  BHaH_defines.h
  BHaH_function_prototypes.h
  main.c
  rhs_eval.c
  initial_data.c
  diagnostics.c
  numerical_grids_and_timestep.c
  cmdline_input_and_parfile_parser.c
  MoL/MoL_step_forward_in_time.c
Generated file sample:
  BHaH_defines.h
  BHaH_function_prototypes.h
  Makefile
  MoL/MoL_free_intermediate_stage_gfs.c
  MoL/MoL_malloc_intermediate_stage_gfs.c
  MoL/MoL_step_forward_in_time.c
  apply_bcs.c
  cmdline_input_and_parfile_parser.c
  commondata_struct_set_to_default.c
  diagnostics.c
  exact_solution_single_Cartesian_point.c
  griddata_free.c
  initial_data.c
  intrinsics/simd_intrinsics.h
  main.c
  numerical_grids_and_timestep.c
  params_struct_set_to_default.c
  progress_indicator.c
  rhs_eval.c
  set_CodeParameters-nopointer.h
  set_CodeParameters-simd.h
  set_CodeParameters.h
  wave_equation_cartesian.par


## Step 5: Connect Files to Responsibilities

The project directory is generated output. Generated C and header files are not hand-edited tutorial source; change the Python generator when the generated implementation must change, then regenerate the project.

In [5]:
responsibilities = {
    "Makefile": "build rules for the standalone executable",
    "wave_equation_cartesian.par": "runtime parameters and defaults",
    "main.c": "program setup, time loop, diagnostics, and cleanup",
    "rhs_eval.c": "right-hand-side evaluation for the wave equation",
    "initial_data.c": "initial values for evolved grid functions",
    "diagnostics.c": "runtime numerical-output writer",
    "cmdline_input_and_parfile_parser.c": "parameter-file parsing and command-line overrides",
    "MoL/MoL_step_forward_in_time.c": "method-of-lines time update",
}

for relpath, role in responsibilities.items():
    print(f"{relpath}: {role}")

Makefile: build rules for the standalone executable
wave_equation_cartesian.par: runtime parameters and defaults
main.c: program setup, time loop, diagnostics, and cleanup
rhs_eval.c: right-hand-side evaluation for the wave equation
initial_data.c: initial values for evolved grid functions
diagnostics.c: runtime numerical-output writer
cmdline_input_and_parfile_parser.c: parameter-file parsing and command-line overrides
MoL/MoL_step_forward_in_time.c: method-of-lines time update


## Step 6: Read the Parameter File and Command-Line Override Path

Runtime parameters live in `wave_equation_cartesian.par` unless a supported command-line argument overrides them. The Cartesian wave-equation executable accepts a convergence-factor override, so `./wave_equation_cartesian 2.0` runs with `convergence_factor = 2.0`.

In [6]:
parfile = PROJECT_DIR / "wave_equation_cartesian.par"
preview_file(parfile, max_lines=16)

parser_source = (PROJECT_DIR / "cmdline_input_and_parfile_parser.c").read_text(encoding="utf-8")
parser_markers = ["convergence_factor", "argv", "read_REAL", "strtod"]
print("Parser source markers:", [token for token in parser_markers if token in parser_source])

wave_equation_cartesian.par
#### wave_equation_cartesian BH@H parameter file. NOTE: only commondata CodeParameters appear here ###
###########################
###########################
### Module: __main__
convergence_factor = 1.0        # (REAL)
diagnostics_output_every = 0.2  # (REAL)
###########################
###########################
### Module: nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData
sigma = 3.0                     # (REAL)
wavespeed = 1.0                 # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.MoLtimestepping.register_all
CFL_FACTOR = 0.5                # (REAL)
t_final = 8.0                   # (REAL)
Parser source markers: ['convergence_factor', 'argv', 'read_REAL', 'strtod']


## Step 7: Inspect the `Makefile`, Build, Clean, and Rebuild

`make` builds the standalone executable. `make clean` removes build products while leaving generated source files in place. After a run, the same target also removes diagnostic text files; Step 8 checks that with a real diagnostic file.

In [7]:
require_tool("make")

makefile = PROJECT_DIR / "Makefile"
preview_file(makefile, max_lines=18)

run_command(["make"], cwd=PROJECT_DIR)
executable = PROJECT_DIR / PROJECT_NAME
print("executable exists:", executable.name)

source_before_clean = PROJECT_DIR / "main.c"
run_command(["make", "clean"], cwd=PROJECT_DIR)
print("clean removed the executable and kept generated sources")

run_command(["make"], cwd=PROJECT_DIR)
print("rebuilt executable:", executable.name)

make: available
Makefile
CC ?= gcc  # assigns the value CC to gcc only if environment variable CC is not already set

CFLAGS = -std=gnu99 -O2 -march=native -g -Wall -I.
CXXFLAGS = -I. -O2 -g -Wall -Wno-unknown-pragmas -march=native
VALGRIND_CFLAGS = -I. -std=gnu99 -O2 -g -Wall -Wno-unknown-pragmas
INCLUDEDIRS =
LDFLAGS = -lm
# Check for OpenMP support
OPENMP_FLAG = -fopenmp
COMPILER_SUPPORTS_OPENMP := $(shell echo | $(CC) $(OPENMP_FLAG) -E - >/dev/null 2>&1 && echo YES || echo NO)

ifeq ($(COMPILER_SUPPORTS_OPENMP), YES)
    CFLAGS += $(OPENMP_FLAG)
    LDFLAGS += $(OPENMP_FLAG)
endif

OBJ_FILES = apply_bcs.o cmdline_input_and_parfile_parser.o commondata_struct_set_to_default.o diagnostics.o exact_solution_single_Cartesian_point.o griddata_free.o initial_data.o main.o MoL/MoL_free_intermediate_stage_gfs.o MoL/MoL_malloc_intermediate_stage_gfs.o MoL/MoL_step_forward_in_time.o numerical_grids_and_timestep.o params_struct_set_to_default.o progress_indicator.o rhs_eval.o

$ make
cwd: .


return code: 0
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c apply_bcs.c -o apply_bcs.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c cmdline_input_and_parfile_parser.c -o cmdline_input_and_parfile_parser.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c commondata_struct_set_to_default.c -o commondata_struct_set_to_default.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c diagnostics.c -o diagnostics.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c exact_solution_single_Cartesian_point.c -o exact_solution_single_Cartesian_point.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c griddata_free.c -o griddata_free.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c initial_data.c -o initial_data.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c main.c -o main.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c MoL/MoL_free_intermediate_stage_gfs.c -o MoL/MoL_free_intermediate_stage_gfs.o
cc -std=gnu9

return code: 0
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c apply_bcs.c -o apply_bcs.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c cmdline_input_and_parfile_parser.c -o cmdline_input_and_parfile_parser.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c commondata_struct_set_to_default.c -o commondata_struct_set_to_default.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c diagnostics.c -o diagnostics.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c exact_solution_single_Cartesian_point.c -o exact_solution_single_Cartesian_point.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c griddata_free.c -o griddata_free.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c initial_data.c -o initial_data.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c main.c -o main.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c MoL/MoL_free_intermediate_stage_gfs.c -o MoL/MoL_free_intermediate_stage_gfs.o
cc -std=gnu9

## Step 8: Run with a Convergence-Factor Override and Inspect Diagnostics

Diagnostic text files provide quick evidence that the executable produced numerical output. This run uses the command-line override `2.0`, which writes `out0d-conv_factor2.00.txt`.

In [8]:
run_command([f"./{PROJECT_NAME}", "2.0"], cwd=PROJECT_DIR, max_lines=20)

diagnostic = PROJECT_DIR / "out0d-conv_factor2.00.txt"
preview_file(diagnostic, max_lines=10)

numeric_rows = [
    line
    for line in diagnostic.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]
print("numeric rows:", len(numeric_rows))

run_command(["make", "clean"], cwd=PROJECT_DIR)
print("final clean removed executable and diagnostic products")

$ ./wave_equation_cartesian 2.0
cwd: .


return code: 0
It: 0 t=0.000 / 8.0 = 0.00% dt=1/12.8 | t/h=0.00 ETA 0h00m00s
It: 1 t=0.078 / 8.0 = 0.98% dt=1/12.8 | t/h=6736.71 ETA 0h00m04s
It: 2 t=0.156 / 8.0 = 1.95% dt=1/12.8 | t/h=6749.77 ETA 0h00m04s
It: 3 t=0.234 / 8.0 = 2.93% dt=1/12.8 | t/h=6706.06 ETA 0h00m04s
It: 4 t=0.312 / 8.0 = 3.91% dt=1/12.8 | t/h=6838.48 ETA 0h00m04s
It: 5 t=0.391 / 8.0 = 4.88% dt=1/12.8 | t/h=6695.79 ETA 0h00m04s
It: 6 t=0.469 / 8.0 = 5.86% dt=1/12.8 | t/h=6779.30 ETA 0h00m03s
It: 7 t=0.547 / 8.0 = 6.84% dt=1/12.8 | t/h=6845.05 ETA 0h00m03s
It: 8 t=0.625 / 8.0 = 7.81% dt=1/12.8 | t/h=6891.44 ETA 0h00m03s
It: 9 t=0.703 / 8.0 = 8.79% dt=1/12.8 | t/h=6935.09 ETA 0h00m03s
...
It: 93 t=7.266 / 8.0 = 90.82% dt=1/12.8 | t/h=7108.67 ETA 0h00m00s
It: 94 t=7.344 / 8.0 = 91.80% dt=1/12.8 | t/h=7110.31 ETA 0h00m00s
It: 95 t=7.422 / 8.0 = 92.77% dt=1/12.8 | t/h=7112.50 ETA 0h00m00s
It: 96 t=7.500 / 8.0 = 93.75% dt=1/12.8 | t/h=7114.38 ETA 0h00m00s
It: 97 t=7.578 / 8.0 = 94.73% dt=1/12.8 | t/h=7111.23 ETA 0h00m00s

## Step 9: What to Edit, What to Regenerate, and Where to Go Next

Treat `project/wave_equation_cartesian/` as generated output. Edit Python generator code when changing generated C or headers, regenerate the project, then rebuild with `make`. Edit the `.par` file or pass supported command-line overrides when changing runtime parameters.

Continue with:

- [NRPy in Ten Minutes](../1-intro/ten_minute_overview.ipynb) for the symbolic-to-code overview.
- [Wave Equation and C Code Generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb) for wave-equation expressions and C code generation.
- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb) for the full Cartesian wave-equation workflow.

## Where This Leads

- [NRPy Package Map](package_map.ipynb) reviews the prerequisite step.
- [NRPy in Ten Minutes](../1-intro/ten_minute_overview.ipynb) continues the tutorial path.
- [Index](../index.ipynb) returns to the full tutorial spine.